# gnomAD AF vs sample frequency correlation plots

This notebook compares how often variants appear in this sequencing run against their reported allele frequencies in gnomAD.

It makes two scatter plots:

1. Overall gnomAD AF vs `sample_fraction`
2. gnomAD `AF_grpmax` vs `sample_fraction`

Rows with missing, `"."`, or non-numeric AF values are excluded from each plot.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

input_csv = "/project/knathans_shared/donetski/Notebooks/OutputFiles/01.4_Variant_frequency_per_sample/Run2/csv/Run2_16_genes_unique_with_variant_frequency.csv"
output_dir = Path(r"/project/knathans_shared/donetski/Notebooks/OutputFiles/12_Frequency_Correlation_Plots")
output_dir.mkdir(exist_ok=True)

df = pd.read_csv(input_csv, encoding="latin1", low_memory=False)

print(df.shape)
df.head()

## Check available AF columns

This lists the gnomAD allele frequency columns so we can confirm genome and exome columns are present.

In [ ]:
genome_af_cols = [
    c for c in df.columns
    if "gnomad41_genome" in c and "AF" in c
]

print("Genome AF columns:")
print(genome_af_cols)

## Choose columns for plotting

We use the overall genome AF and the maximum ancestry-group genome AF.

In [ ]:
af_plot_cols = [
    "gnomad41_genome_AF",
    "gnomad41_genome_AF_grpmax",
]

af_plot_cols = [c for c in af_plot_cols if c in df.columns]

print("Columns used for plotting:")
print(af_plot_cols)

## Convert values to numeric

Missing values such as `"."` are converted to `NaN` and excluded from the plots.

In [ ]:
for col in af_plot_cols + ["sample_fraction"]:
    df[col] = pd.to_numeric(df[col].replace(".", pd.NA), errors="coerce")

## Make scatter plots

Each plot shows Pearson and Spearman correlation between the selected gnomAD genome AF column and `sample_fraction`.

In [ ]:
def scatter_corr(af_col):
    plot_df = df[[af_col, "sample_fraction"]].dropna()

    pearson = plot_df[af_col].corr(plot_df["sample_fraction"], method="pearson")
    spearman = plot_df[af_col].corr(plot_df["sample_fraction"], method="spearman")

    plt.figure(figsize=(7, 5))
    plt.scatter(plot_df[af_col], plot_df["sample_fraction"], alpha=0.6)

    plt.xlabel(af_col)
    plt.ylabel("sample_fraction")
    plt.title(
        f"{af_col} vs sample_fraction\n"
        f"Pearson={pearson:.3f}, Spearman={spearman:.3f}, n={len(plot_df)}"
    )

    out_png = output_dir / f"{af_col}_vs_sample_fraction.png"
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.show()

    print("Saved:", out_png)

In [ ]:
for col in af_plot_cols:
    scatter_corr(col)